# 실험 06 — 모델 표현 능력에 따른 암종 분류 오류 비교

**연구 질문:** 동일한 유전자 변이 유무(0/1) 입력에서도 선형 모델과 비선형 모델은 서로 다른 암종을 혼동하는가?

- 비교: **SGD (log_loss), Random Forest, XGBoost, LightGBM**.
- 데이터: **학습용 Train 데이터 6,201건**, 유전자 4,384개. **공식 Test 데이터는 열거나 사용하지 않음.**
- 평가: 중복 이진 프로파일을 같은 폴드에 묶는 **3-Fold StratifiedGroupKFold**; 공식 지표와 동일한 **Macro F1**을 주지표로 사용.
- 분석: 전체 및 암종별 성능, KIPAN↔KIRC / GBMLGG↔LGG / BRCA↔OV 혼동, **같은 환자가 모델 간에도 반복 오분류되는지**.
- 해석 주의: 모델 비교만으로 '데이터 자체가 부족하다'는 인과적 결론을 증명할 수 없음. 학습 및 평가 기준을 동일하게 맞춰 **모델 표현 능력의 영향**을 조사하는 실험.

> 이전 실험 05와 동일한 Train 자료를 쓰되, 실험 06은 **새롭게 정의한 동일 3-Fold 내에서 SGD를 다시 학습**하여 공정하게 비교합니다. 이 결과를 기존 단일 홀드아웃 실험과 직접 동일 기준으로 비교하면 안 됩니다.


## 0. 환경 준비

VS Code에서 커널을 선택하고 아래 셀을 순서대로 실행하세요. 최초 1회 설치가 필요하면 다음 줄의 주석을 해제해 실행하세요.

```python
%pip install numpy pandas scipy scikit-learn xgboost lightgbm matplotlib
```

프로젝트 폴더 구조:

```text
cancer_project/
├── 06_model_comparison.ipynb
├── outputs/
│   ├── X_train_binary.npz
│   ├── y_train.npy
│   └── gene_columns.json  # 선택
└── experiment_06/        # 자동 생성
```


In [ ]:
from pathlib import Path
import hashlib, json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

SEED = 42
N_SPLITS = 3
# 빠른 실행 점검 전용: True일 때는 1개 Fold와 적은 트리 수만 실행함.
# 정식 실험 결과가 필요하면 반드시 False로 실행!
QUICK_CHECK = False
N_JOBS = 4  # PC 사양에 따라 2~8로 조정

ROOT = Path.cwd()
DATA_DIR = ROOT / 'outputs'
OUT_DIR = ROOT / 'experiment_06'
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert (DATA_DIR / 'X_train_binary.npz').is_file(), f'파일 없음: {DATA_DIR / "X_train_binary.npz"}'
assert (DATA_DIR / 'y_train.npy').is_file(), f'파일 없음: {DATA_DIR / "y_train.npy"}'
print('프로젝트:', ROOT.resolve())
print('출력 폴더:', OUT_DIR.resolve())

## 1. Train에서 만든 전처리 결과 불러오기

In [ ]:
X = sparse.load_npz(DATA_DIR / 'X_train_binary.npz').tocsr().astype(np.float32)
X.sum_duplicates()
X.eliminate_zeros()
X.sort_indices()
y_text = np.load(DATA_DIR / 'y_train.npy', allow_pickle=True).astype(str)
assert len(y_text) == X.shape[0]
assert X.shape[1] == 4384, '예상 유전자 수(4,384개)와 다릅니다. 전처리 파일을 확인하세요.'
assert np.isfinite(X.data).all(), '입력 행렬에 유효하지 않은 값이 있습니다.'
assert np.isin(X.data, [0.0, 1.0]).all(), '이번 실험 입력은 0/1 이진 특징이어야 합니다.'
le = LabelEncoder()  # 공식 Test 라벨을 사용하지 않음; Train의 암종으로만 고정
le.fit(y_text)
y = le.transform(y_text)
classes = le.classes_
assert len(classes) == 26
print(f'전체 Train: {X.shape[0]:,}명 / 특징 {X.shape[1]:,}개 / 클래스 {len(classes)}개')
print('전처리 데이터만 읽었습니다. Test는 사용하지 않았습니다.')
display(pd.Series(y_text).value_counts().rename_axis('암종').reset_index(name='환자 수'))

## 2. 동일 프로파일을 그룹으로 묶어 3-Fold 교차검증

- 이미 0/1 변이 유무로 인코딩된 CSR 행렬에서 활성 유전자의 **정렬된 인덱스**를 해시합니다.
- 동일 이진 프로파일이 서로 다른 폴드의 Train/Validation에 동시에 들어가지 않도록 합니다.
- 이는 평가 분리 방지를 위한 보수적 그룹 정의이며 환자 동일인 여부를 나타내지 않습니다.
- **검증 폴드별 클래스가 26개 모두 있는지** 확인합니다.


In [ ]:
profile_hashes = []
for i in range(X.shape[0]):
    indices = X.indices[X.indptr[i]:X.indptr[i+1]]
    profile_hashes.append(hashlib.sha256(indices.tobytes()).digest())

# pd.factorize(list) 호환성 문제를 피하도록 NumPy 배열 사용
groups = pd.factorize(np.array(profile_hashes, dtype='|S32'))[0]
print('고유 이진 프로파일:', len(np.unique(groups)), '/ 중복 행:', len(groups)-len(np.unique(groups)))

cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
all_folds = list(cv.split(X, y, groups))
for fold, (tr, va) in enumerate(all_folds, 1):
    assert set(groups[tr]).isdisjoint(set(groups[va])), '중복 프로파일 누출 발생'
    assert len(np.unique(y[tr])) == len(classes), f'Fold {fold} 학습에 암종 누락'
    assert len(np.unique(y[va])) == len(classes), f'Fold {fold} 검증에 암종 누락'
    print(f'Fold {fold}: Train {len(tr):,}명 / Valid {len(va):,}명 / 검증 클래스 {len(np.unique(y[va]))}개')
folds = all_folds[:1] if QUICK_CHECK else all_folds
if QUICK_CHECK:
    print('주의: QUICK_CHECK=True: 임시 실행 결과는 정식 연구 결과가 아닙니다.')

## 3. 비교 모델 설정

입력 데이터와 검증 분할은 동일합니다. 모델별 하이퍼파라미터는 **사전에 고정한 비교용 값**이며 최적화된 값이 아닙니다.

- **SGD:** 희소한 고차원 특징에 적합한 선형 기준 모델. 클래스 균형 가중치를 적용.
- **Random Forest:** 비선형 분할과 유전자 상호작용을 학습할 수 있는 앙상블.
- **XGBoost / LightGBM:** 부스팅 기반 비선형 모델. 학습 시간이 길 수 있음.

주의: 모델별 `class_weight` 또는 샘플 가중치의 처리 방식은 정확히 같지 않을 수 있으므로, 이번 실험 결과는 **모델 구현·설정에 따른 전체적인 차이**입니다. 모델 구조만의 순수 인과효과가 아닙니다.


In [ ]:
def build_model(name, quick=False):
    trees = 35 if quick else 220
    if name == 'SGD':
        return SGDClassifier(loss='log_loss', alpha=1e-4, class_weight='balanced',
                             max_iter=1500, tol=1e-3, random_state=SEED)
    if name == 'RandomForest':
        return RandomForestClassifier(n_estimators=trees, max_features='sqrt',
                                      min_samples_leaf=2, class_weight='balanced_subsample',
                                      random_state=SEED, n_jobs=N_JOBS)
    if name == 'XGBoost':
        return XGBClassifier(n_estimators=trees, max_depth=5, learning_rate=0.08,
                             subsample=0.85, colsample_bytree=0.8,
                             objective='multi:softprob', tree_method='hist',
                             eval_metric='mlogloss', random_state=SEED,
                             n_jobs=N_JOBS)
    if name == 'LightGBM':
        return LGBMClassifier(n_estimators=trees, learning_rate=0.06,
                              num_leaves=23, min_child_samples=18,
                              colsample_bytree=0.8, subsample=0.85, subsample_freq=1,
                              class_weight='balanced', verbosity=-1,
                              random_state=SEED, n_jobs=N_JOBS)
    raise ValueError(name)

MODEL_NAMES = ['SGD', 'RandomForest', 'XGBoost', 'LightGBM']
PAIRS = [('KIPAN','KIRC'),('GBMLGG','LGG'),('BRCA','OV')]
print('비교 모델:', ', '.join(MODEL_NAMES))

## 4. 모델 학습 / Macro F1 / 검증 예측 저장

- 이 셀은 컴퓨터 성능에 따라 실행 시간이 길 수 있습니다.
- 각 모델과 Fold가 끝날 때마다 `experiment_06/cv_metrics_progress.csv`, `experiment_06/oof_predictions_progress.csv`를 갱신합니다.
- 외부 Test 데이터는 읽지 않으며, 검증 폴드의 특징을 기반으로 인코더·스케일러 등을 학습하지 않습니다.


In [ ]:
metric_rows, prediction_rows = [], []
for fold, (tr, va) in enumerate(folds, start=1):
    Xtr, Xva = X[tr], X[va]
    ytr, yva = y[tr], y[va]
    print(f'\n===== Fold {fold}/{len(folds)} =====', flush=True)
    for name in MODEL_NAMES:
        model = build_model(name, quick=QUICK_CHECK)
        start = time.perf_counter()
        model.fit(Xtr, ytr)
        pred = model.predict(Xva).astype(int)
        elapsed = time.perf_counter() - start
        row = {
            'fold': fold, 'model': name, 'n_train': len(tr), 'n_valid': len(va),
            'macro_f1': f1_score(yva, pred, labels=np.arange(len(classes)), average='macro', zero_division=0),
            'weighted_f1': f1_score(yva, pred, labels=np.arange(len(classes)), average='weighted', zero_division=0),
            'accuracy': accuracy_score(yva, pred), 'elapsed_sec': round(elapsed, 2),
        }
        metric_rows.append(row)
        for original_idx, a, b in zip(va, yva, pred):
            prediction_rows.append({
                'row_index': int(original_idx), 'fold': fold, 'model': name,
                'true': classes[a], 'pred': classes[b], 'correct': int(a==b),
                'mutation_count': int(X.indptr[original_idx+1]-X.indptr[original_idx]),
            })
        pd.DataFrame(metric_rows).to_csv(OUT_DIR/'cv_metrics_progress.csv',index=False,encoding='utf-8-sig')
        pd.DataFrame(prediction_rows).to_csv(OUT_DIR/'oof_predictions_progress.csv',index=False,encoding='utf-8-sig')
        print(f'{name:13s} Macro F1={row["macro_f1"]:.4f}, Accuracy={row["accuracy"]:.4f}, {elapsed:.1f}s',flush=True)

metrics = pd.DataFrame(metric_rows)
oof = pd.DataFrame(prediction_rows)
assert not oof.duplicated(['row_index','model']).any()
if not QUICK_CHECK:
    for name in MODEL_NAMES:
        assert oof[oof.model==name].row_index.nunique()==len(y), '검증 예측 누락 발견'
display(metrics)

## 5. 같은 Fold에서 모델별 비교 — 평균과 변동성

In [ ]:
summary = (metrics.groupby('model', as_index=False)
           .agg(macro_f1_mean=('macro_f1','mean'), macro_f1_std=('macro_f1','std'),
                weighted_f1_mean=('weighted_f1','mean'), accuracy_mean=('accuracy','mean'),
                elapsed_sec_total=('elapsed_sec','sum'))
           .sort_values('macro_f1_mean',ascending=False))
print('교차검증 Macro F1 평균 (정식 실행에서는 3-Fold)')
display(summary.round(4))
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(summary.model, summary.macro_f1_mean, yerr=summary.macro_f1_std.fillna(0), capsize=4)
ax.set_ylabel('Macro F1 (mean ± SD)')
ax.set_ylim(0, max(0.6, float(summary.macro_f1_mean.max())*1.2))
ax.set_title('Experiment 06: model comparison')
plt.xticks(rotation=20)
plt.tight_layout(); plt.savefig(OUT_DIR/'model_comparison.png',dpi=160); plt.show()

## 6. 암종별 F1 비교

모든 모델에 같은 클래스 순서로 F1을 계산합니다. `class_f1_by_fold.csv`는 Fold별 기록, `class_f1_summary.csv`는 모델·암종별 Fold 평균입니다.


In [ ]:
class_rows=[]
for (fold,name), df in oof.groupby(['fold','model']):
    report = classification_report(df['true'], df['pred'], labels=list(classes),
                                   output_dict=True, zero_division=0)
    for cls in classes:
        class_rows.append({'fold':fold,'model':name,'cancer':cls,
                           'support':int(report[cls]['support']),
                           'precision':report[cls]['precision'],
                           'recall':report[cls]['recall'], 'f1':report[cls]['f1-score']})
class_df = pd.DataFrame(class_rows)
class_summary = (class_df.groupby(['model','cancer'],as_index=False)
                 .agg(f1_mean=('f1','mean'), recall_mean=('recall','mean'),
                      precision_mean=('precision','mean'), total_support=('support','sum')))
pivot = class_summary.pivot(index='cancer',columns='model',values='f1_mean').round(3)
print('암종별 3-Fold F1 평균:')
display(pivot)

## 7. 주요 혼동 암종 쌍 비교

분모가 서로 다른 암종 간 혼동을 함께 비교할 때는 절대 건수만 보지 않고, 각 실제 암종에서 상대 암종으로 잘못 예측한 **방향별 비율**도 확인합니다.


In [ ]:
pair_rows=[]
for (fold,name), df in oof.groupby(['fold','model']):
    for a,b in PAIRS:
        for src,dst in ((a,b),(b,a)):
            ss=df[df['true']==src]
            n=int((ss['pred']==dst).sum())
            pair_rows.append({'fold':fold,'model':name,'pair':f'{a} ↔ {b}',
                              'true':src,'predicted':dst,'support':len(ss),
                              'errors':n,'error_rate':n/len(ss) if len(ss) else np.nan})
pair_df=pd.DataFrame(pair_rows)
pair_summary=(pair_df.groupby(['model','pair'],as_index=False)
              .agg(errors=('errors','sum'),support=('support','sum')))
pair_summary['pair_error_per_support']=pair_summary.errors / pair_summary.support
print('두 방향을 합친 오분류 수 및 상대적 오류율:')
display(pair_summary.sort_values(['pair','model']))

## 8. 같은 환자가 여러 모델에서도 반복 오분류되는가?

`hard_cases.csv`는 검증 예측에서 **네 모델 모두 틀린 환자**와 일부 모델만 틀린 환자를 식별합니다. 실제 환자 식별 정보는 저장하지 않고 Train 행 번호만 기록합니다.

- 모델들이 공통으로 실패한다고 해서 **데이터 자체가 본질적으로 분류 불가능하다는 증거는 아닙니다**.
- 모델 성능 차이는 학습 조건과 하이퍼파라미터의 영향도 받을 수 있습니다.


In [ ]:
wide = oof.pivot(index='row_index',columns='model',values='correct')
wide = wide.dropna(subset=MODEL_NAMES)  # QUICK_CHECK에서도 분석 가능
n_correct = wide[MODEL_NAMES].astype(int).sum(axis=1)
base_rows = oof[oof.model=='SGD'].set_index('row_index')[['true','mutation_count']]
hard = base_rows.loc[wide.index].copy()
hard['n_models_correct']=n_correct
hard['n_models_wrong']=len(MODEL_NAMES)-n_correct
hard['all_wrong']=hard.n_models_correct.eq(0)
hard['all_correct']=hard.n_models_correct.eq(len(MODEL_NAMES))
hard['row_index']=hard.index
print('모델별 정답 수 분포 (0=4개 모델 모두 오답):')
display(hard.n_models_correct.value_counts().sort_index().rename_axis('맞힌 모델 수').reset_index(name='환자 수'))
print('모든 모델에서 틀린 환자 수:',int(hard.all_wrong.sum()), '/',len(hard))
print('암종별 공통 오분류 비율 (상위 10개):')
display(hard.groupby('true').agg(n=('all_wrong','size'), common_errors=('all_wrong','sum'),
                                 common_error_rate=('all_wrong','mean')).sort_values('common_error_rate',ascending=False).head(10))

## 9. 변이 부담에 따른 모델 공통 실패

변이 수에 따라 공통 오분류가 달라지는지 *기술통계*로 살펴봅니다. 구간은 사전 정의한 `1~7`, `8~27`, `28+`, `0`으로 표시하며, 암종 구성에 따른 혼란변수 가능성을 별도로 고려합니다.


In [ ]:
bins=[-1,0,7,27,np.inf]
labels=['0개','1~7개','8~27개','28개 이상']
hard['burden_group']=pd.cut(hard.mutation_count,bins=bins,labels=labels)
burden_summary=(hard.groupby('burden_group',observed=False)
                .agg(n=('all_wrong','size'), all_wrong_n=('all_wrong','sum'),
                     all_wrong_rate=('all_wrong','mean')).reset_index())
display(burden_summary)
print('주의: 0개 변이 환자처럼 동일 프로파일이 한 폴드에 몰린 경우, Fold별 해석에 주의하세요.')

## 10. 결과 저장 및 실험 해석

정식 결과인 경우 `experiment_06` 폴더를 압축해서 공유해 주세요. 빠른 점검 모드의 결과는 별도 표시하며, 정식 실험 결과로 인용하지 마세요.

**실험의 판단 기준**

1. SGD보다 비선형 모델이 Macro F1을 일관되게 높이는가?
2. KIPAN↔KIRC 등 주요 암종 쌍의 혼동이 실제로 감소하는가?
3. 모델이 바뀌어도 반복적으로 틀리는 환자가 있는가?
4. 그 환자들이 특정 암종·변이 수 구간에 집중되는가?

**한계:** 이번 실험에서는 XAI나 임상적 타당성을 검증하지 않으며 Test 데이터로 전처리·튜닝을 하지 않습니다. 후속 단계에서 모델 선택과 하이퍼파라미터 탐색의 반복적 검증셋 과적합에 주의해야 합니다.


In [ ]:
metrics.to_csv(OUT_DIR/'cv_metrics.csv',index=False,encoding='utf-8-sig')
summary.to_csv(OUT_DIR/'cv_summary.csv',index=False,encoding='utf-8-sig')
oof.to_csv(OUT_DIR/'oof_predictions.csv',index=False,encoding='utf-8-sig')
class_df.to_csv(OUT_DIR/'class_f1_by_fold.csv',index=False,encoding='utf-8-sig')
class_summary.to_csv(OUT_DIR/'class_f1_summary.csv',index=False,encoding='utf-8-sig')
pair_df.to_csv(OUT_DIR/'pair_confusion_by_fold.csv',index=False,encoding='utf-8-sig')
pair_summary.to_csv(OUT_DIR/'pair_confusion_summary.csv',index=False,encoding='utf-8-sig')
hard.reset_index(drop=True).to_csv(OUT_DIR/'hard_cases.csv',index=False,encoding='utf-8-sig')
burden_summary.to_csv(OUT_DIR/'burden_common_errors.csv',index=False,encoding='utf-8-sig')
config={'seed':SEED,'n_splits':N_SPLITS,'quick_check':QUICK_CHECK,
        'feature':'binary_gene_mutation','models':MODEL_NAMES,
        'evaluation':'StratifiedGroupKFold grouped by identical binary profiles',
        'test_data_used':False,'n_samples':len(y),'n_features':X.shape[1]}
(OUT_DIR/'experiment_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
print('완료! 결과:', OUT_DIR.resolve())
if QUICK_CHECK: print('주의: QUICK_CHECK=True 결과입니다. 정식 결과를 원하면 False로 변경 후 전체 셀 재실행!')